# Week 5 Exercise: Expert Knowledge Worker (RAG)

This notebook implements a **RAG (Retrieval Augmented Generation)** pipeline for Insurellm, an Insurance Tech company. It combines learnings from all 5 days:

- **Day 1**: Simple RAG with keyword-based context retrieval
- **Day 2**: Document chunking, vector embeddings (Chroma), and visualization
- **Day 3**: Full RAG pipeline with LangChain (retriever + LLM)
- **Day 4**: Evaluation (retrieval metrics + LLM-as-judge)
- **Day 5**: Advanced techniques (optional: reranking, query rewriting)

### Requirements
- Run from the **week5** directory: `cd week5 && jupyter notebook`
- Or run from repo root — the setup cell will configure paths automatically
- Ensure `.env` has `OPENAI_API_KEY` (and optionally `HF_TOKEN` for HuggingFace embeddings)

## Setup

In [ ]:
# Ensure we're in the week5 directory so implementation, evaluation, and knowledge-base resolve
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
if (repo_root / "week5").exists():
    week5_dir = repo_root / "week5"
else:
    week5_dir = repo_root  # Already in week5
sys.path.insert(0, str(week5_dir))
os.chdir(week5_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import glob
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
import gradio as gr

load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key found (starts with {openai_api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set — add it to .env")

## Part A: Document Ingestion & Chunking

Load documents from the knowledge base and split them into chunks for embedding.

In [ ]:
# Load documents from knowledge-base (employees, products, contracts, company)
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# Chunk documents using RecursiveCharacterTextSplitter (Day 2 style)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
print(f"Sample chunk:\n{chunks[0].page_content[:300]}...")

## Part B: Vector Store (Chroma)

Encode chunks into vectors and store in Chroma. Uses HuggingFace `all-MiniLM-L6-v2` (free) or OpenAI embeddings.

In [ ]:
# Use HuggingFace embeddings (free) — or OpenAI for higher quality
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Rebuild vector store (delete existing if present)
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=DB_NAME
)
print(f"Vector store created with {vectorstore._collection.count()} documents")

### Optional: Visualize vectors (Day 2 style)

Reduce embeddings to 2D/3D with t-SNE and plot by document type.

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

collection = vectorstore._collection
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
doc_types = [m.get("doc_type", "unknown") for m in result["metadatas"]]
colors = {"products": "blue", "employees": "green", "contracts": "red", "company": "orange"}
color_list = [colors.get(t, "gray") for t in doc_types]

tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(vectors)
fig = go.Figure(data=[go.Scatter(
    x=reduced[:, 0], y=reduced[:, 1], mode="markers",
    marker=dict(size=5, color=color_list, opacity=0.8),
    text=doc_types, hoverinfo="text"
)])
fig.update_layout(title="2D Vector Store Visualization", width=700, height=500)
fig.show()

## Part C: RAG Pipeline (Day 3)

Connect retriever and LLM to answer questions using retrieved context.

In [ ]:
retriever = vectorstore.as_retriever(k=10)
llm = ChatOpenAI(temperature=0, model_name=MODEL)

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing Insurellm.
Use the given context to answer questions. If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history=None):
    history = history or []
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for h in history:
        if h.get("role") == "user":
            messages.append(HumanMessage(content=h["content"]))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs

In [ ]:
# Test the RAG pipeline
answer, docs = answer_question("Who founded Insurellm?")
print("Q: Who founded Insurellm?")
print(f"A: {answer}")
print(f"\nRetrieved {len(docs)} chunks")

## Part D: Evaluation (Day 4)

Run retrieval and answer evaluation on a sample of test questions. The full evaluation module (`evaluation.eval`) uses `implementation.answer` — for that, ensure you've run `implementation/ingest.py` from the week5 directory. Here we evaluate using our notebook's RAG directly.

In [ ]:
# Evaluate on a few test questions from the evaluation suite
from evaluation.test import load_tests

tests = load_tests()
sample = tests[:3]  # First 3 tests

for t in sample:
    answer, docs = answer_question(t.question)
    print(f"Q: {t.question}")
    print(f"A: {answer}")
    print(f"Reference: {t.reference_answer}")
    print("-" * 60)

## Part E: Gradio Chat Interface

Interactive Q&A interface for the Insurellm Expert Assistant.

In [ ]:
def chat_fn(message, history):
    history = history or []
    # Convert Gradio history to list of dicts (handle both message format and tuple format)
    prior = []
    for h in history:
        if isinstance(h, dict):
            prior.append({"role": h.get("role", "user"), "content": h.get("content", str(h))})
        elif isinstance(h, (list, tuple)) and len(h) >= 2:
            prior.append({"role": "user", "content": h[0]})
            prior.append({"role": "assistant", "content": h[1]})
    answer, _ = answer_question(message, prior)
    return answer

gr.ChatInterface(chat_fn, title="Insurellm Expert Assistant", type="messages").launch(inbrowser=True)